In [ ]:
import csv
import time
import re
import threading
from queue import Queue
from urllib.parse import quote
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# ============================================================
#  并行配置
# ============================================================
NUM_PRODUCERS = 2
NUM_CONSUMERS = 10
TARGET_CATEGORIES = ["数据分析", "数据科学", "算法", "风控"]
OUTPUT_FILE   = "bytedance_campus_threaded.csv"

# ============================================================
# 工具函数
# ============================================================

def get_test_driver():
    options = webdriver.ChromeOptions()
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option('useAutomationExtension', False)
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-gpu')
    options.add_argument('--disable-dev-shm-usage')
    options.page_load_strategy = 'eager'
    
    #driver_path = "C:/Program Files/Google/Chrome/Application/chromedriver.exe"
    #service = Service(executable_path=driver_path)
    driver = webdriver.Chrome(options=options)
    driver.set_page_load_timeout(30)
    return driver


FIELD_NAMES = ["职位名称", "职位ID", "工作城市", "全职实习", "细分方向", "招聘类型",
               "职位简述", "职位要求", "详情页URL", "搜索类别"]

def parse_detail_page(driver, item_title):
    job_data = {}
    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, '.block-content'))
        )
        try:
            job_title = driver.find_element(By.CSS_SELECTOR, '.positionItem-title-text').text.strip()
        except:
            job_title = item_title
        
        job_id = "未知"; location = "未知"; job_type = "未知"
        job_category_detail = "未知"; recruitment_type = "未知"
        
        try:
            job_info = driver.find_element(By.CSS_SELECTOR, '.job-info')
            spans = job_info.find_elements(By.XPATH, './span')
            spans_text = [s.text.strip() for s in spans if s.text.strip()]
            tags = [t for t in spans_text if "职位 ID" not in t]
            if len(tags) >= 1: location = tags[0]
            if len(tags) >= 2: job_type = tags[1]
            if len(tags) >= 3: job_category_detail = tags[2]
            if len(tags) >= 4: recruitment_type = tags[3]
            for s in spans_text:
                if "职位 ID" in s:
                    job_id = s.replace("职位 ID：", "").replace("职位 ID:", "").strip()
        except: pass
        
        jd_text = "未知"; job_requirements = "未知"
        try:
            for bt in driver.find_elements(By.CSS_SELECTOR, '.block-title'):
                title_text = bt.text.strip()
                content_els = bt.find_elements(By.XPATH, './following-sibling::div[contains(@class, "block-content")]')
                if content_els:
                    content_text = driver.execute_script("return arguments[0].textContent", content_els[0]).strip()
                    if "职位描述" in title_text: jd_text = content_text
                    elif "职位要求" in title_text: job_requirements = content_text
        except: pass
        
        job_data = {"职位名称": job_title, "职位ID": job_id, "工作城市": location,
                    "全职实习": job_type, "细分方向": job_category_detail,
                    "招聘类型": recruitment_type, "职位简述": jd_text, "职位要求": job_requirements}
    except:
        job_data = {"职位名称": item_title, "职位ID": "解析失败", "工作城市": "未知",
                    "全职实习": "未知", "细分方向": "未知", "招聘类型": "未知",
                    "职位简述": "解析失败", "职位要求": "解析失败"}
    return job_data


def split_categories(categories, num_producers):
    chunks = []
    base = len(categories) // num_producers
    remainder = len(categories) % num_producers
    start = 0
    for i in range(num_producers):
        size = base + (1 if i < remainder else 0)
        chunks.append(categories[start:start + size])
        start += size
    return [c for c in chunks if c]


# ============================================================
# 生产者
# ============================================================

def producer_worker(categories, task_queue, stats, producer_id):
    driver = get_test_driver()
    total = 0
    try:
        driver.get("https://jobs.bytedance.com/campus/position")
        time.sleep(3)
        
        for cat in categories:
            tprint(f"[P{producer_id}] 开始收集: {cat}")
            
            try:
                search_input = WebDriverWait(driver, 15).until(
                    EC.presence_of_element_located((By.XPATH, '//input[contains(@placeholder, "输入城市或职位进行搜索") or contains(@placeholder, "搜索职位")]'))
                )
                search_input.clear()
                search_input.send_keys(Keys.CONTROL, "a"); search_input.send_keys(Keys.DELETE)
                time.sleep(0.5)
                search_input.send_keys(cat); search_input.send_keys(Keys.ENTER)
            except:
                continue
            
            time.sleep(4)
            current_url = driver.current_url
            current_match = re.search(r'[&?]current=(\d+)', current_url)
            page = int(current_match.group(1)) if current_match else 1
            
            while True:
                cards = driver.find_elements(By.CSS_SELECTOR, 'div[class*="positionItem"]')
                if not cards: break
                
                for card in cards:
                    try:
                        title_el = card.find_element(By.CSS_SELECTOR, '.positionItem-title-text')
                        title = title_el.text.strip()
                        a_tag = card.find_element(By.XPATH, './parent::a')
                        href = a_tag.get_attribute("href")
                        task_queue.put({"title": title, "url": href, "category": cat})
                        total += 1
                    except:
                        continue
                
                page += 1
                next_url = re.sub(r'([&?])current=\d+', rf'\1current={page}', current_url)
                driver.get(next_url)
                time.sleep(3)
                
            with print_lock:
                stats['enqueued'] = total
                
    except Exception as e:
        import traceback; tprint(f"[P{producer_id}] 异常: {e}"); traceback.print_exc()
    finally:
        driver.quit()
    
    tprint(f"[P{producer_id}] 完成，入队 {total} 条")


# ============================================================
# 消费者
# ============================================================

def consumer_worker(task_queue, csv_writer, file_lock, stats, consumer_id):
    driver = get_test_driver()
    processed = 0
    try:
        while True:
            item = task_queue.get()
            if item is None:
                task_queue.task_done()
                break
            
            try:
                driver.get(item["url"])
                job_data = parse_detail_page(driver, item["title"])
                job_data["详情页URL"] = item["url"]
                job_data["搜索类别"] = item.get("category", "")
                
                with file_lock:
                    csv_writer.writerow(job_data)
                    stats['written'] += 1
                processed += 1
            except:
                pass
            finally:
                task_queue.task_done()
    except Exception as e:
        import traceback; tprint(f"[C{consumer_id}] 异常: {e}"); traceback.print_exc()
    finally:
        driver.quit()
    
    tprint(f"[C{consumer_id}] 退出，处理 {processed} 条")


# ============================================================
# 统计报告 & 打印
# ============================================================

print_lock = threading.Lock()

def tprint(msg):
    with print_lock:
        print(msg)

def stats_reporter(stats, stop_event):
    while not stop_event.is_set():
        stop_event.wait(10)
        if stop_event.is_set(): break
        with print_lock:
            print(f"[STATS] 入队 {stats['enqueued']} | 已写 {stats['written']} | 队列 ~{stats['enqueued'] - stats['written']}")


# ============================================================
# 主函数
# ============================================================

def run_parallel_crawl():
    producer_assignments = split_categories(TARGET_CATEGORIES, NUM_PRODUCERS)
    
    task_queue = Queue()
    stats = {'enqueued': 0, 'written': 0}
    stop_event = threading.Event()
    
    csv_file = open(OUTPUT_FILE, 'w', newline='', encoding='utf-8-sig')
    csv_writer = csv.DictWriter(csv_file, fieldnames=FIELD_NAMES)
    csv_writer.writeheader(); csv_file.flush()
    file_lock = threading.Lock()
    
    # 统计报告线程
    reporter = threading.Thread(target=stats_reporter, args=(stats, stop_event), daemon=True)
    reporter.start()
    
    # 生产者
    producers = []
    for pid, cats in enumerate(producer_assignments, 1):
        t = threading.Thread(target=producer_worker, args=(cats, task_queue, stats, pid), daemon=True)
        t.start(); producers.append(t)
    
    tprint(f"生产者已启动 ({len(producers)} 线程)")
    time.sleep(5)
    
    # 消费者
    consumers = []
    for cid in range(1, NUM_CONSUMERS + 1):
        t = threading.Thread(target=consumer_worker, args=(task_queue, csv_writer, file_lock, stats, cid), daemon=True)
        t.start(); consumers.append(t)
    
    tprint(f"消费者已启动 ({NUM_CONSUMERS} 线程)")
    tprint("=" * 50)
    
    for t in producers: t.join()
    
    tprint("=" * 50)
    tprint("生产者全部完成，等待消费者处理剩余任务...")
    
    for _ in range(NUM_CONSUMERS): task_queue.put(None)
    for t in consumers: t.join()
    
    stop_event.set()
    csv_file.close()
    
    tprint(f"全部完成，入队 {stats['enqueued']} 条，写入 {stats['written']} 条")
    return OUTPUT_FILE


if __name__ == "__main__":
    result_file = run_parallel_crawl()
    try:
        df = pd.read_csv(result_file, encoding='utf-8-sig')
        if len(df) > 0:
            df.drop_duplicates(subset=['职位ID', '详情页URL'], inplace=True)
            df.to_csv(result_file, index=False, encoding='utf-8-sig')
            print(f"\n去重后共计 {len(df)} 条 → {result_file}")
        else:
            print("\n未能获取到任何数据")
    except Exception as e:
        print(f"\n读取结果文件失败: {e}")

生产者已启动 (2 线程)，开始收集 URL...
消费者已启动 (10 线程)，开始消费...
[Producer 1] 开始收集类别: 数据分析
[Producer 2] 开始收集类别: 算法
[Producer 1] 数据分析 p1 → +10 条入队 | 累计 10
[Producer 2] 算法 p1 → +10 条入队 | 累计 10
[Producer 1] 数据分析 p2 → +10 条入队 | 累计 20
[Producer 2] 算法 p2 → +10 条入队 | 累计 20
[Producer 1] 数据分析 p3 → +10 条入队 | 累计 30
[Producer 2] 算法 p3 → +10 条入队 | 累计 30
[Producer 1] 数据分析 p4 → +10 条入队 | 累计 40
[Producer 2] 算法 p4 → +10 条入队 | 累计 40
[Producer 1] 数据分析 p5 → +10 条入队 | 累计 50
[Producer 2] 算法 p5 → +10 条入队 | 累计 50
[Producer 1] 数据分析 p6 → +10 条入队 | 累计 60
[Producer 2] 算法 p6 → +10 条入队 | 累计 60
[Producer 1] 数据分析 p7 → +10 条入队 | 累计 70
[Producer 2] 算法 p7 → +10 条入队 | 累计 70
[Producer 1] 数据分析 p8 → +10 条入队 | 累计 80
[Producer 2] 算法 p8 → +10 条入队 | 累计 80
[Producer 1] 数据分析 p9 → +10 条入队 | 累计 90
[Producer 2] 算法 p9 → +10 条入队 | 累计 90
[Producer 1] 数据分析 p10 → +10 条入队 | 累计 100
[Producer 2] 算法 p10 → +10 条入队 | 累计 100
[Producer 1] 数据分析 p11 → +10 条入队 | 累计 110
[Producer 2] 算法 p11 → +10 条入队 | 累计 110
[Consumer 9] 已处理 10 条
[Consumer 4] 已处理 10 条
[Producer 1] 